# USF FIBO Extraction Demo

**Pipeline**: Docling → LangExtract (FIBO few-shot) → ArcadeDB → PROV-O

This notebook demonstrates the full unstructured ingestion pipeline on a synthetic banking regulatory document.

In [ ]:
# Install deps if needed (uncomment in fresh env)
# !pip install langextract docling chonkie[semantic] rdflib pyyaml loguru httpx pandas matplotlib

In [ ]:
import sys, os, json, pathlib, textwrap
from pathlib import Path

# Add repo root to path
REPO_ROOT = Path(os.getcwd()).parents[1]
sys.path.insert(0, str(REPO_ROOT / 'apps' / 'ingest'))

print('Repo root:', REPO_ROOT)

## 1. Create synthetic banking regulatory text

We use synthetic text representative of an EBA regulatory disclosure document.

In [ ]:
SYNTHETIC_BANKING_TEXT = """
ANNUAL PRUDENTIAL DISCLOSURE REPORT
Northgate Financial Group PLC — For the year ending 31 December 2023
Prepared in accordance with CRR Article 431-455 and EBA Guidelines on Disclosure

1. LEGAL ENTITY INFORMATION

Northgate Financial Group PLC (LEI: 213800ABCDEF12345678) is the ultimate parent entity
of the Group, incorporated in England and Wales. Its principal subsidiary, Northgate Bank Ltd
(LEI: 213800ZYXWVU98765432), is authorised and regulated by the Prudential Regulation
Authority (PRA) and the Financial Conduct Authority (FCA).

2. CAPITAL ADEQUACY

As at 31 December 2023, the Group maintained a Common Equity Tier 1 (CET1) ratio of 14.8%
(2022: 13.9%), well above the regulatory minimum of 4.5%. Total regulatory capital amounted
to GBP 2,847,000,000.

3. LARGE EXPOSURES

The following counterparties each represent an exposure exceeding 10% of Tier 1 capital:

- Deutsche Bank AG (LEI: 7LTWFZYICNSX8D621K86): gross exposure EUR 420,000,000,
  net exposure after collateral EUR 285,000,000. Credit rating: A (Fitch).
- Société Générale S.A. (LEI: O2RNE8IBXP4R0TD8PU41): gross exposure EUR 310,000,000,
  net exposure EUR 198,000,000. Credit rating: A- (Fitch).
- HSBC Holdings PLC: gross exposure USD 275,000,000.

4. CREDIT RISK

Total on-balance-sheet credit exposures as at the reporting date were GBP 18,450,000,000,
comprising:
  - Corporate loans: GBP 7,200,000,000 (NPL ratio 1.8%)
  - Retail mortgages: GBP 6,800,000,000 (NPL ratio 0.6%)
  - SME lending: GBP 2,100,000,000 (NPL ratio 3.2%)
  - Other financial institutions: GBP 2,350,000,000

5. KEY TRANSACTIONS IN 2023

On 15 March 2023, the Group completed a EUR 500,000,000 Senior Non-Preferred bond issuance
(ISIN: XS2600000001) at a coupon of 5.25%, maturing March 2028. Joint bookrunners were
Barclays Bank PLC and Citigroup Global Markets Limited.

On 1 July 2023, Northgate Bank Ltd entered into a GBP 200,000,000 revolving credit facility
with a syndicate led by Lloyds Bank plc, with a tenor of 3 years and margin of SONIA + 95bps.

6. REMUNERATION

Total variable remuneration awarded to identified staff in 2023: GBP 45,200,000.
Number of identified staff: 287 individuals.
"""

# Write to a temp text file to simulate a document
import tempfile
tmp_doc = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
tmp_doc.write(SYNTHETIC_BANKING_TEXT)
tmp_doc.close()
DOC_PATH = tmp_doc.name
print(f'Synthetic document saved to: {DOC_PATH}')
print(f'Characters: {len(SYNTHETIC_BANKING_TEXT):,}')

## 2. Docling Parser — document structure

In [ ]:
# For plain text, skip docling and simulate the DocumentResult
# In production, DoclingParser handles PDF/DOCX/HTML

from usf_ingest.pipelines.unstructured.docling_parser import DocumentResult, SectionElement

# Simulate parsed document structure
# (Real run: document = await DoclingParser().parse(DOC_PATH))
document = DocumentResult(
    source_path=DOC_PATH,
    mime_type='text/plain',
    text=SYNTHETIC_BANKING_TEXT,
    sections=[
        SectionElement(heading='LEGAL ENTITY INFORMATION', level=1, char_start=202, char_end=260),
        SectionElement(heading='CAPITAL ADEQUACY', level=1, char_start=570, char_end=610),
        SectionElement(heading='LARGE EXPOSURES', level=1, char_start=800, char_end=840),
        SectionElement(heading='CREDIT RISK', level=1, char_start=1200, char_end=1240),
        SectionElement(heading='KEY TRANSACTIONS IN 2023', level=1, char_start=1600, char_end=1650),
    ],
    page_count=3,
)

print(f'Document: {document.source_path}')
print(f'Words: {document.word_count:,}')
print(f'Sections found: {len(document.sections)}')
print()
for s in document.sections:
    print(f'  H{s.level}: {s.heading} (chars {s.char_start}–{s.char_end})')

## 3. Semantic Chunking

In [ ]:
# Simulate chunking (real run: chunks = await SemanticChunker().chunk(document.text))
from usf_ingest.pipelines.unstructured.chunker import Chunk

# Split by double newline as a simple simulation
paragraphs = [p.strip() for p in SYNTHETIC_BANKING_TEXT.split('\n\n') if p.strip()]
simulated_chunks = []
offset = 0
for i, para in enumerate(paragraphs):
    pos = SYNTHETIC_BANKING_TEXT.find(para, offset)
    simulated_chunks.append(Chunk(
        text=para,
        char_start=pos,
        char_end=pos + len(para),
        chunk_index=i,
    ))
    offset = pos + len(para)

print(f'Chunks produced: {len(simulated_chunks)}')
print()
for c in simulated_chunks[:4]:
    print(f'  Chunk {c.chunk_index} [{c.char_start}:{c.char_end}] ({len(c.text)} chars)')
    print(f'    Preview: {c.text[:120].strip()}...')
    print()

## 4. LangExtract with FIBO Few-Shot

In production this calls the real LangExtract API with Gemini/OpenAI.
Here we simulate the output to show the data structure.

In [ ]:
# Load few-shot examples
FEW_SHOT_DIR = REPO_ROOT / 'packages' / 'ontologies' / 'fibo' / 'few_shot'

few_shot_files = list(FEW_SHOT_DIR.glob('*.json'))
print(f'Few-shot files found: {len(few_shot_files)}')
for f in few_shot_files:
    examples = json.loads(f.read_text())
    total_extractions = sum(len(e['extractions']) for e in examples)
    print(f'  {f.name}: {len(examples)} examples, {total_extractions} total extractions')

In [ ]:
from usf_ingest.pipelines.unstructured.langextract_runner import ExtractionResult

# Simulate LangExtract output — in production:
# runner = LangExtractRunner(config={'llm_provider': 'gemini'})
# results = await runner.extract(document.text, ontology_module='fibo')

simulated_extractions = [
    ExtractionResult('LegalEntity', 'Northgate Financial Group PLC', 'fibo:LegalEntity',
        {'lei': '213800ABCDEF12345678', 'role': 'parent_entity', 'jurisdiction': 'England and Wales'},
        char_interval=(93, 127), confidence_score=0.97, model_id='gemini-2.5-flash'),
    ExtractionResult('LegalEntity', 'Northgate Bank Ltd', 'fibo:LegalEntity',
        {'lei': '213800ZYXWVU98765432', 'role': 'subsidiary', 'regulator': 'PRA/FCA'},
        char_interval=(248, 267), confidence_score=0.96, model_id='gemini-2.5-flash'),
    ExtractionResult('LegalEntity', 'Deutsche Bank AG', 'fibo:LegalEntity',
        {'lei': '7LTWFZYICNSX8D621K86', 'credit_rating': 'A', 'rating_agency': 'Fitch'},
        char_interval=(800, 816), confidence_score=0.98, model_id='gemini-2.5-flash'),
    ExtractionResult('LegalEntity', 'Société Générale S.A.', 'fibo:LegalEntity',
        {'lei': 'O2RNE8IBXP4R0TD8PU41', 'credit_rating': 'A-', 'rating_agency': 'Fitch'},
        char_interval=(920, 942), confidence_score=0.97, model_id='gemini-2.5-flash'),
    ExtractionResult('LegalEntity', 'HSBC Holdings PLC', 'fibo:LegalEntity',
        {'role': 'large_exposure_counterparty'},
        char_interval=(1050, 1068), confidence_score=0.95, model_id='gemini-2.5-flash'),
    ExtractionResult('Counterparty', 'Deutsche Bank AG', 'fibo:Counterparty',
        {'gross_exposure_eur': '420000000', 'net_exposure_eur': '285000000'},
        char_interval=(800, 816), confidence_score=0.94, model_id='gemini-2.5-flash'),
    ExtractionResult('Transaction', 'EUR 500,000,000 Senior Non-Preferred bond issuance', 'fibo:Transaction',
        {'isin': 'XS2600000001', 'coupon': '5.25', 'currency': 'EUR', 'amount': '500000000', 'maturity': '2028-03'},
        char_interval=(1610, 1680), confidence_score=0.96, model_id='gemini-2.5-flash'),
    ExtractionResult('Transaction', 'GBP 200,000,000 revolving credit facility', 'fibo:Transaction',
        {'currency': 'GBP', 'amount': '200000000', 'tenor_years': '3', 'margin_bps': '95'},
        char_interval=(1750, 1800), confidence_score=0.93, model_id='gemini-2.5-flash'),
    ExtractionResult('Account', 'revolving credit facility with a syndicate led by Lloyds Bank plc', 'fibo:Account',
        {'account_type': 'RevolvingCreditFacility', 'currency': 'GBP', 'amount': '200000000'},
        char_interval=(1750, 1830), confidence_score=0.91, model_id='gemini-2.5-flash'),
    ExtractionResult('LegalEntity', 'unverified entity without span', 'fibo:LegalEntity',
        {}, char_interval=None, confidence_score=0.3, model_id='gemini-2.5-flash'),  # Will be quarantined
]

print(f'Simulated extractions: {len(simulated_extractions)}')
for e in simulated_extractions:
    grounded = '✓' if e.char_interval else '✗ ungrounded'
    print(f'  [{e.ontology_class}] "{e.text_span[:50]}" conf={e.confidence_score:.2f} {grounded}')

## 5. Confidence Filter

In [ ]:
from usf_ingest.pipelines.unstructured.confidence_filter import ConfidenceFilter

flt = ConfidenceFilter(confidence_threshold=0.5)
passed, quarantined = flt.filter(simulated_extractions, job_id='demo-job-001')

print(f'Passed:      {len(passed)}')
print(f'Quarantined: {len(quarantined)}')
print()
if quarantined:
    print('Quarantine records:')
    for q in quarantined:
        print(f'  ["{q.extraction.text_span[:60]}"] → {q.reason}')

## 6. Entity Types Distribution

In [ ]:
import collections
import matplotlib.pyplot as plt

type_counts = collections.Counter(e.ontology_class for e in passed)

fig, ax = plt.subplots(figsize=(8, 4))
labels = [c.split(':')[-1] for c in type_counts.keys()]
values = list(type_counts.values())
ax.bar(labels, values, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'][:len(labels)])
ax.set_title('Extracted Entity Types (FIBO)', fontsize=14)
ax.set_xlabel('FIBO Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('entity_types_distribution.png', dpi=150)
plt.show()
print('Entity type counts:', dict(type_counts))

## 7. Confidence Score Distribution

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        'text_span': e.text_span[:40],
        'ontology_class': e.ontology_class.split(':')[-1],
        'confidence_score': e.confidence_score,
        'char_start': e.char_interval[0] if e.char_interval else None,
        'char_end': e.char_interval[1] if e.char_interval else None,
        'model_id': e.model_id,
    }
    for e in passed
])

print('Extraction results:')
display(df)

print(f'\nMean confidence: {df["confidence_score"].mean():.3f}')
print(f'Min confidence:  {df["confidence_score"].min():.3f}')
print(f'Max confidence:  {df["confidence_score"].max():.3f}')

## 8. ArcadeDB Nodes (Simulated)

In production: `await ArcadeDBBuilder(client).build_knowledge_graph(passed, job_id='demo-001')`

In [ ]:
from usf_ingest.pipelines.unstructured.arcadedb_builder import _make_iri

# Show the IRIs that would be created
print('ArcadeDB nodes that would be created:')
print()
inserted_iris = []
for e in passed:
    iri = _make_iri(e.ontology_class, e.text_span)
    inserted_iris.append(iri)
    vertex_type_map = {
        'fibo:LegalEntity': 'LegalEntity',
        'fibo:Account': 'FinancialAccount',
        'fibo:Transaction': 'FinancialTransaction',
        'fibo:Counterparty': 'Counterparty',
    }
    vtype = vertex_type_map.get(e.ontology_class, 'Entity')
    print(f'  MERGE (:{vtype} {{iri: "{iri[:60]}..."}})')
    print(f'    label: "{e.text_span[:50]}", confidence: {e.confidence_score}')
    print()

print(f'Total nodes: {len(inserted_iris)}')

## 9. PROV-O Block

Every extraction result generates provenance triples linking entities to the extraction activity.

In [ ]:
from datetime import datetime, timezone

job_id = 'demo-job-001'
model_id = 'gemini-2.5-flash'
now = datetime.now(timezone.utc).isoformat()

prov_turtle = f"""
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix usf: <https://usf.io/ontology/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# Extraction Activity
<https://usf.io/provenance/activity/{job_id}>
    a prov:Activity ;
    prov:startedAtTime "{now}"^^xsd:dateTime ;
    usf:jobId "{job_id}" ;
    usf:modelId "{model_id}" ;
    usf:ontologyModule "fibo" ;
    prov:wasAssociatedWith <https://usf.io/provenance/agent/langextract/{model_id}> .

# Software Agent
<https://usf.io/provenance/agent/langextract/{model_id}>
    a prov:SoftwareAgent ;
    rdfs:label "LangExtract/{model_id}" .

# Sample entity provenance triples
"""

for e, iri in zip(passed[:3], inserted_iris[:3]):
    prov_turtle += f"""
<{iri}>
    prov:wasGeneratedBy <https://usf.io/provenance/activity/{job_id}> ;
    usf:charStart {e.char_interval[0]} ;
    usf:charEnd {e.char_interval[1]} ;
    usf:confidenceScore "{e.confidence_score}"^^xsd:decimal .
"""

print(prov_turtle)

# Validate with rdflib
try:
    from rdflib import Graph
    g = Graph()
    g.parse(data=prov_turtle, format='turtle')
    print(f'✓ PROV-O Turtle is valid RDF — {len(g)} triples')
except Exception as ex:
    print(f'RDF parse error: {ex}')

## 10. Pipeline Summary

In [ ]:
summary = {
    'source': DOC_PATH,
    'ontology_module': 'fibo',
    'document_words': document.word_count,
    'sections': len(document.sections),
    'chunks': len(simulated_chunks),
    'total_extractions': len(simulated_extractions),
    'passed_filter': len(passed),
    'quarantined': len(quarantined),
    'kg_nodes_created': len(inserted_iris),
    'entity_types': dict(type_counts),
    'mean_confidence': round(df['confidence_score'].mean(), 3),
    'model_id': 'gemini-2.5-flash',
    'prov_activity': f'https://usf.io/provenance/activity/{job_id}',
}

print('Pipeline Summary')
print('=' * 50)
for k, v in summary.items():
    print(f'  {k:<30} {v}')